<picture>
  <source media="(prefers-color-scheme: dark)" srcset="https://raw.githubusercontent.com/thinkube/meepsquared/main/assets/meepsquared-logo-dark.svg">
  <img alt="Meep²" src="https://raw.githubusercontent.com/thinkube/meepsquared/main/assets/meepsquared-logo-light.svg" width="239" height="80">
</picture>

# 05 · Several GPUs without NVLink

> **Licence: Apache-2.0.** Copyright Alejandro Martínez Corriá and the Thinkube contributors.
> This notebook, [meepsquared](https://github.com/thinkube/meepsquared) and every package meepsquared installs are permissively licensed:
> you can use, change and ship what you build with them, commercially or not. See [Licensing](https://thinkube.org/meepsquared/licensing.html).

**Where it runs.** On any Linux machine with an NVIDIA GPU of the Ampere generation or later and Meep² installed ([Install](https://github.com/thinkube/meepsquared#install)). The model and the data come from the Hugging Face Hub. Run the cells from top to bottom.

A few cells are marked **Thinkube only**. [Thinkube](https://thinkube.org) is the open-source AI platform Meep² is developed on; on a Thinkube cluster these cells read the model from the cluster's copy, record the run and serve the result. Elsewhere they do nothing, so run every cell as it comes.

The outputs saved in this notebook come from a run on a Thinkube cluster, on two NVIDIA RTX 3090, so they show the cluster's model paths and its run links.

## What you will do

Fine-tune on the GPUs of one machine: consumer GPUs over plain PCIe, no NVLink. The question a second GPU answers depends on the model, and so does the mode to use.

### Three ways to use several GPUs

| Mode | When | What it does |
|---|---|---|
| `ddp` | the model fits one GPU | a copy of the model per GPU, each training its own batch; the adapters' gradients are averaged after every step. Twice the batch per step, the fastest when it fits: 1.8× one GPU on two RTX 3090 |
| `fsdp` | the model does not fit one GPU | every layer's weights sharded across the GPUs (FSDP2); each layer is gathered when it runs and released after. A BF16 model loads on the CPU first so each GPU takes only its share; a QLoRA model loads on each GPU |
| `split` | the model does not fit one GPU | half the layers on each GPU, one process, the activations handed from one GPU to the next; one GPU busy at a time. Simpler than FSDP2 and, without NVLink, 1.5× faster: it moves activations once per layer, FSDP2 moves the weights |

**DDP** (distributed data parallel) is for speed: the model is small enough, and you want more tokens per second. **FSDP2** and **split** are for size: the model does not fit, and two GPUs make the training possible at all. A model that fits one GPU gains nothing from sharding, only communication.

### How the training runs

`torchrun` starts one process per GPU for `ddp` and `fsdp`; the processes find each other, each loads its share, and they synchronise every step. `split` is one ordinary process. So the training is a script, `sft_several_gpus.py` next to this notebook, and the notebook starts it and shows its output. The script takes the same data as notebook 01: a Hub dataset, a Parquet or a JSONL file of conversations.

On Thinkube, a notebook job on a node that shares its GPUs takes every GPU of the node by asking for all of them (`gpus` = the node's GPU count).

A training job (`m2.TrainingJob`, notebooks 01 and 02) does not cover several GPUs yet; this notebook and its script use meepsquared's lower-level calls.

Needs `meepsquared`; on Thinkube, the `meepsquared` kernel has it.

## Settings

Every choice of the notebook is a name in this cell, so you can rerun the notebook with other values without touching the cells below, and a notebook job can override any of them with its parameters.

The defaults are the DDP case: Llama 3.1 8B in BF16 fits a 24 GB GPU, so two GPUs double the batch. For the other case, set `MODEL` to a model that does not fit one of your GPUs, such as `Qwen/Qwen3-14B` in BF16 (29.5 GB) on 24 GB GPUs, and `MODE` to `"split"` or `"fsdp"`.

In [2]:
PARAMETERS = globals().get("PARAMETERS", {})

MODEL_SOURCE = PARAMETERS.get("MODEL_SOURCE", "huggingface")      # "huggingface"; "thinkube" only on a Thinkube cluster
MODEL = PARAMETERS.get("MODEL", "meta-llama/Llama-3.1-8B-Instruct")
MODE = PARAMETERS.get("MODE", "ddp")                         # "ddp", "fsdp" or "split"
METHOD = PARAMETERS.get("METHOD", "lora")                    # "lora" or "qlora"
GPUS = PARAMETERS.get("GPUS", None)                          # processes for ddp and fsdp; None for every GPU seen
DATASET = PARAMETERS.get("DATASET", "HuggingFaceH4/no_robots")
ROWS = PARAMETERS.get("ROWS", 480)
EPOCHS = PARAMETERS.get("EPOCHS", None)
STEPS = PARAMETERS.get("STEPS", 30)
BATCH = PARAMETERS.get("BATCH", 8)                           # conversations per GPU per step
MAX_LENGTH = PARAMETERS.get("MAX_LENGTH", 1024)
OUTPUT_DIR = PARAMETERS.get("OUTPUT_DIR", "out/05-several-gpus")
# The folder of sft_several_gpus.py: this notebook's folder, unless a job runs a copy elsewhere
NOTEBOOKS = PARAMETERS.get("NOTEBOOKS")

## Where the model comes from

A fine-tuning starts from a **base model**: a model somebody already trained on a great deal of text, whose behaviour you will adjust with your own data. The base model is the largest thing a training handles, 8 to 70 GB of weights for the models in these notebooks, so where it comes from matters.

`MODEL_SOURCE` chooses:

* `"huggingface"`: the model is downloaded from the Hugging Face Hub by its id, once, into the Hub's cache on this machine. A *gated* model, such as Llama, needs its licence accepted on its Hub page and the token of your account in the `HF_TOKEN` variable.
* `"thinkube"`, only on a [Thinkube](https://thinkube.org) cluster: the model comes from the cluster's **model mirror**: a copy made once into the cluster's shared storage, which every GPU node reads. Nothing is downloaded by a training, a gated model needs no token here, and a 70 GB model loads in about a minute. A model is mirrored in Thinkube Control (AI > Models). `thinkube_models.mirrored_model_path` finds its folder, and stops with an error naming the model if it is not mirrored, before any time is spent.

In [3]:
import os
from pathlib import Path

if MODEL_SOURCE == "thinkube":
    import thinkube_models as tkm

    MODEL_PATH = tkm.mirrored_model_path(MODEL)
    print(f"{MODEL}: {MODEL_PATH}")
elif MODEL_SOURCE == "huggingface":
    MODEL_PATH = MODEL
    print(f"{MODEL}: from the Hugging Face Hub")
else:
    raise ValueError(f"MODEL_SOURCE must be 'thinkube' or 'huggingface', not {MODEL_SOURCE!r}")

meta-llama/Llama-3.1-8B-Instruct: /home/thinkube/thinkube/mlflow/artifacts/1/97bcb5e9c2514970be7de1bb085f7189/artifacts/model


## The GPUs

What is there, and how they are connected. `nvidia-smi topo -m` prints the link between each pair of GPUs: `NV#` is NVLink, `PIX`/`PXB`/`PHB` are PCIe at increasing distance, `SYS` crosses CPU sockets. Over PCIe, the data between GPUs moves at a few tens of GB/s, a tenth of NVLink: that is why the choice of mode matters here.

In [4]:
import subprocess
import sys

import torch

NOTEBOOKS = Path(NOTEBOOKS) if NOTEBOOKS else Path.cwd()
SCRIPT = NOTEBOOKS / "sft_several_gpus.py"
if not SCRIPT.is_file():
    raise FileNotFoundError(f"{SCRIPT} is not there: run this notebook from meepsquared's notebooks folder, or set NOTEBOOKS")
count = torch.cuda.device_count()
print(f"{count} GPUs:", ", ".join(f"{torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 2**30:.0f} GiB)" for i in range(count)))
print(subprocess.run(["nvidia-smi", "topo", "-m"], capture_output=True, text=True).stdout.split("Legend")[0])
GPUS = GPUS or count
if GPUS > count:
    raise RuntimeError(f"GPUS={GPUS}, but {count} are visible")

2 GPUs: NVIDIA GeForce RTX 3090 (24 GiB), NVIDIA GeForce RTX 3090 (24 GiB)
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-31	0		N/A
GPU1	PHB	 X 	0-31	0		N/A




## Train

The cell builds the command and runs the script, showing its training lines as they come. Open `sft_several_gpus.py` to see what each mode changes; it is short. The parts worth knowing:

* `ddp` needs nothing special: `torchrun` sets the process group, and TRL's trainer wraps the model.
* `fsdp` loads a BF16 model on the CPU (`device_map={"": "cpu"}`) so the GPUs receive only their shards, uses meepsquared's `fsdp_config(model)` to tell Accelerate which layers to shard, and the PyTorch optimizer: bitsandbytes' 8-bit optimizers cannot update sharded parameters.
* `split` loads with `device_map="balanced"`, which spreads the layers over the GPUs, and trains like one GPU.
* Checkpointing is `"auto"` only for `ddp`: meepsquared's measurement looks at one GPU's memory, which says nothing about a sharded or spread model, so the other modes checkpoint every layer.
* Saving is the same call as in the one-GPU notebooks, `save_pretrained_merged`, made by every process; the first one writes. Under `fsdp` the adapter's weights are sharded, so all the processes gather them first, which is why each of them has to make the call.

At the end the script prints one `RESULT` line with the steps, the time, the first and last loss and the peak memory, and where the adapter is.

In [5]:
import os

script = [str(SCRIPT), "--mode", MODE, "--model", MODEL_PATH, "--dataset", DATASET, "--batch", str(BATCH),
          "--max-length", str(MAX_LENGTH), "--output", OUTPUT_DIR]
script += ["--rows", str(ROWS)] if ROWS else []
script += ["--epochs", str(EPOCHS)] if EPOCHS else ["--steps", str(STEPS)]
script += ["--qlora"] if METHOD == "qlora" else []
if MODE == "split":
    command = [sys.executable, "-u", *script]
else:
    command = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node", str(GPUS), *script]
print(" ".join(command[-len(script) - 3:]), flush=True)
process = subprocess.Popen(command, cwd=NOTEBOOKS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                           env={**os.environ, "TOKENIZERS_PARALLELISM": "false"})
for line in process.stdout:
    if "'loss'" in line or line.startswith(("RESULT", "adapter saved", "meepsquared")) or "Error" in line[:80]:
        print(line, end="", flush=True)
if process.wait() != 0:
    raise RuntimeError(f"the training ended with exit code {process.returncode}; its output is above")

torch.distributed.run --nproc_per_node 2 /home/thinkube/thinkube/notebooks/meepsquared-release/sft_several_gpus.py --mode ddp --model /home/thinkube/thinkube/mlflow/artifacts/1/97bcb5e9c2514970be7de1bb085f7189/artifacts/model --dataset HuggingFaceH4/no_robots --batch 8 --max-length 1024 --output out/05-several-gpus --rows 480 --steps 30
meepsquared: checkpointing 27 of 32 layers (one layer for the data's 8 longest examples, 7,941 tokens (8 examples, end to end): the bound, as the trainer orders its batches itself: 0.88 GiB keeping everything, 0.70 selective, 0.07 recomputed whole; its forward and backward 128 / 139 / 186 ms in that order; free: 8 of 24 GiB)
meepsquared: checkpointing 27 of 32 layers (one layer for the data's 8 longest examples, 7,941 tokens (8 examples, end to end): the bound, as the trainer orders its batches itself: 0.88 GiB keeping everything, 0.70 selective, 0.07 recomputed whole; its forward and backward 129 / 141 / 187 ms in that order; free: 8 of 24 GiB)
{'loss'

## Use the adapter

The adapter the script saved is the same kind of folder as the one-GPU notebooks produce, whatever the mode. It loads onto the base model with PEFT: meepsquared's `from_pretrained` of the base model, `PeftModel.from_pretrained` of the adapter, then `for_inference`. Notebook 01 shows the merged save and, on Thinkube, the registration for serving.

## Where to go next

* [Several GPUs](https://thinkube.org/meepsquared/several-gpus.html) in the documentation: when each mode, the settings behind `fsdp_config`, QLoRA under FSDP2, loading a large model without a copy per process.
* The [benchmark](https://thinkube.org/meepsquared/benchmarks.html#_two_rtx_3090_without_nvlink) on two RTX 3090: Llama 8B and Qwen 27B QLoRA under DDP, Qwen3-14B under FSDP2 and split.